# Task 6: Label Free Scorer

In [1]:
#@title Clone Repository (if using Google Colab)

import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab.")
    # 1. Step back to root first to prevent getcwd directory errors
    %cd -q /content

    # 2. Clear previous clones safely
    !rm -rf /content/Accenture-1O-contract-review-challenge

    # 3. Clone Task 3 branch
    print("Cloning repo...")
    !git clone -b task6-runs-and-scorer https://github.com/Break-Through-Tech/Accenture-1O-contract-review-challenge.git

    # 4. Move into the working directory
    %cd -q /content/Accenture-1O-contract-review-challenge/notebooks/

    print("\nCurrent Branch: ")

    !git branch --show-current
else:
    print("\nRunning in local environment.")

Running in Google Colab.
Cloning repo...
Cloning into 'Accenture-1O-contract-review-challenge'...
remote: Enumerating objects: 830, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 830 (delta 149), reused 133 (delta 101), pack-reused 603 (from 2)
Receiving objects: 100% (830/830), 119.98 MiB | 9.32 MiB/s, done.
Resolving deltas: 100% (407/407), done.
Updating files: 100% (152/152), done.

Current Branch: 
task6-runs-and-scorer


In [2]:
from pathlib import Path
import json

config_path = Path("october_task1_outputs/task1_frozen_model_specification.json")
with open(config_path, 'r', encoding='utf-8') as file:
    frozen_config = json.load(file)

MODEL_ID = frozen_config["encoder"]["primary_model_id"]
MODEL_NAME = frozen_config["encoder"]["primary_name"]
SEEDS = frozen_config["training"]["random_seeds"]
MAX_LENGTH = frozen_config["tokenization"]["max_length"]

FINAL_IDS = frozen_config["categories"]

OUTPUT_DIR = Path("october_task6_outputs"); OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Model: {frozen_config["encoder"]["primary_name"]}")
print(f"Seeds: {SEEDS}")
print(f"Categories: {FINAL_IDS}")

Model: Legal-BERT
Seeds: [42, 123, 2026]
Categories: ['governing_law', 'renewal_term', 'revenue_profit_sharing', 'cap_on_liability', 'uncapped_liability', 'termination_for_convenience', 'anti_assignment', 'audit_rights', 'license_grant', 'exclusivity']


In [3]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/october_task4_outputs')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load Data

We use the weighted model as it was discovered to perform better in our evaluation (See Task 8). We'll use seed 42 as the default seed to stay consistent with baseline results.

In [4]:
import pandas as pd

MODEL_DIR = DRIVE_DIR / 'full_weighted' / 'final_weighted_Legal-BERT_seed42'
RUN_DIR = DRIVE_DIR / 'run_weighted_Legal-BERT_seed42' / 'checkpoint-2034'
TOKENIZER_DIR = RUN_DIR

# get thresholds from frozen
THRESHOLD_DIR = Path('october_task8_outputs/task8_frozen_thresholds.csv')
thresholds = pd.read_csv(THRESHOLD_DIR)
thr42 = thresholds.loc[thresholds["seed"] == 42, ["category", "threshold"]]
assert not thr42["category"].duplicated().any()
threshold42 = {c: float(t) for c, t in zip(thr42["category"], thr42["threshold"])}

# CUAD Label-Free Scorer

We'll be using the CUADClassifier defined in `cuad_classifier.py`:

```py
class CUADClassifier:
  def __init__(...):
    ...

  def from_run(cls, root, seed, thresholds, categories, **kwargs):
    ...
  
  ...
```

In [6]:
from cuad_classifier import CUADClassifier
import json

model_root = DRIVE_DIR / 'full_weighted'

clf = CUADClassifier.from_run(model_root, seed=42, thresholds=thresholds, categories=FINAL_IDS)

sentence = "This Agreement shall be governed by the laws of California."

score = clf.score(sentence, batch_size=1)
print(json.dumps(score[0][0], indent=2))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{
  "category_id": "governing_law",
  "score": 0.9923903346061707,
  "threshold": 0.6500000000000001,
  "predicted": true
}


Confirm that the output aligns with the saved logits from Task 4 (that were used in Task 8's evaluation). The difference is minimal and is plausible considering the float trunctation that took place in the Trainer.

In [7]:
saved = np.load(model_root / "weighted_Legal-BERT_seed42_val_scores.npz", allow_pickle=True)

ids = list(saved["chunk_ids"])  # the order the saved logits are in
chunks = pd.read_parquet("task12b_outputs/test_chunks.parquet")
val_texts = chunks.set_index("chunk_id").loc[ids, "text"].tolist()

cuad_cls = clf.predict_logits(val_texts)

print(np.abs(cuad_cls - saved["val_logits"]).max())

try:
  assert np.allclose(cuad_cls, saved["val_logits"], atol=1e-4)
except AssertionError:
  diff = np.abs(cuad_cls - saved["val_logits"])
  print("max:", diff.max(), "| mean:", diff.mean())
  print("rows with diff > 1e-3:", (diff.max(axis=1) > 1e-3).sum(), "of", len(diff))
  print("per-category max diff:", diff.max(axis=0).round(4))

  # do the *decisions* agree at the frozen thresholds?
  thr = np.array([clf.thresholds[c] for c in FINAL_IDS])
  p_ours = 1 / (1 + np.exp(-cuad_cls))
  p_saved = 1 / (1 + np.exp(-saved["val_logits"]))
  print("flipped decisions:", ((p_ours >= thr) != (p_saved >= thr)).sum(), "of", p_ours.size)

0.008872509
max: 0.008872509 | mean: 0.00090919016
rows with diff > 1e-3: 2752 of 2873
per-category max diff: [0.0057 0.0068 0.0089 0.0047 0.0042 0.0082 0.0057 0.0076 0.005  0.0058]
flipped decisions: 0 of 28730


# Label-Free Run
Get the desired output for chunks:

```
stage_output = {
    "model_version": "example_model_v1",
    "chunk_predictions": [
        {
            "chunk_id": "document_0001__chunk_000",
            "document_id": "document_0001",
            "contract_id": "contract_0001",
            "context_group_id": "context_hash_abc123",
            "chunk_text": "This Agreement is governed by the laws of California.",
            "char_start": 0,
            "char_end": 53,
            "category_results": [
                {
                    "category_id": "governing_law",
                    "score": 0.93,
                    "threshold": 0.70,
                    "predicted": True,
                },
            ],
        },
    ],
}
```

## Validation Set

In [18]:
from utils import load_split_data

documents = load_split_data("validation")["documents"]
df = chunks.merge(
    documents[["contract_id", "document_id"]],
    on="contract_id", how="left", validate="many_to_one",
)

# merge safety checks
assert len(df) == len(chunks), "merge duplicated or dropped chunks"
assert df["document_id"].notna().all(), "some chunks have no document_id"

val_chunks = df[["chunk_id", "document_id", "contract_id", "context_group_id",
                 "text", "char_start", "char_end"]]
print(len(val_chunks))   # expect 2873 for the validation split

val_out = clf.score_chunks(val_chunks)
val_preds = val_out["chunk_predictions"]

assert len(val_preds) == len(val_chunks) == 2873
assert len({p["chunk_id"] for p in val_preds}) == len(val_preds)               # unique chunks
assert all([r["category_id"] for r in p["category_results"]] == FINAL_IDS for p in val_preds)  # order + exactly 10
assert sum(len(p["category_results"]) for p in val_preds) == 28730

2873


## Training Set

In [34]:
train_chunks = chunks = pd.read_parquet("task12b_outputs/train_chunks.parquet")
documents = load_split_data("train")["documents"]
df = chunks.merge(
    documents[["contract_id", "document_id"]],
    on="contract_id", how="left", validate="many_to_one",
)

# merge safety checks
assert len(df) == len(chunks), "merge duplicated or dropped chunks"
assert df["document_id"].notna().all(), "some chunks have no document_id"

train_chunks = df[["chunk_id", "document_id", "contract_id", "context_group_id",
                 "text", "char_start", "char_end"]]
print(len(train_chunks))   # expect 10841 for the training split

train_out = clf.score_chunks(train_chunks)
train_preds = train_out["chunk_predictions"]

assert len(train_preds) == len(train_chunks) == 10841
assert len({p["chunk_id"] for p in train_preds}) == len(train_preds)               # unique chunks
assert all([r["category_id"] for r in p["category_results"]] == FINAL_IDS for p in train_preds)  # order + exactly 10
assert sum(len(p["category_results"]) for p in train_preds) == 108410

10841


In [35]:
# save to october_task6_output_dir
# with open(OUTPUT_DIR / "val_chunk_scores.json", "w", encoding="utf-8") as f:
#     json.dump(val_out, f, indent=4)

with open(OUTPUT_DIR / "train_chunk_scores.json", "w", encoding="utf-8") as f:
  json.dump(train_out, f, indent=4)